# Conditional DDPM: known-family conditioning and equal-weight scenario ensemble

This notebook implements the conditional diffusion experiment used in the study, including conditional training, the SimPEG baseline, DPS inversion, known-family (oracle) inference, and equal-weight scenario pooling.


In [ ]:
# Cell 1 — Imports and configuration
import os, sys, glob, json, math, time, zipfile, io, contextlib, gc, random, warnings, subprocess
try:
    import simpeg, discretize
except ImportError:
    subprocess.check_call([sys.executable,"-m","pip","install","-q","simpeg>=0.25,<0.26","discretize>=0.11","choclo"])

import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
from scipy.stats import spearmanr, wilcoxon, ttest_rel
from discretize import TensorMesh
from simpeg import maps, data, data_misfit, regularization, optimization, inverse_problem, directives, inversion
from simpeg.potential_fields import gravity

warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED=1234
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark=True

# Existing simplified-rich pipeline
BASE_DIR="/kaggle/working/simplified_rich_pipeline"
DATA_DIR=f"{BASE_DIR}/data"
UNCONDITIONAL_MODEL_PATH=f"{BASE_DIR}/training/best.pt"

# New conditional experiment
COND_DIR="/kaggle/working/conditional_rich_pipeline"
TRAIN_DIR=f"{COND_DIR}/training"
RESULTS_DIR=f"{COND_DIR}/results"
CASE_DIR=f"{RESULTS_DIR}/cases"
for p in [TRAIN_DIR,RESULTS_DIR,CASE_DIR]: os.makedirs(p,exist_ok=True)

FORCE_RETRAIN_CONDITIONAL=False
FORCE_REBENCHMARK=False

GRID_SIZE=64
T_STEPS=1000
BASE_CHANNELS=32
TIME_DIM=128
LABEL_DIM=128

EPOCHS=200
BATCH_SIZE=64
LEARNING_RATE=2e-4
WEIGHT_DECAY=1e-6
NUM_WORKERS=2
CHECKPOINT_EVERY=10
SAMPLE_EVERY=20

N_TEST_PER_SPLIT=100
TOTAL_POSTERIOR_SAMPLES=24
SAMPLES_PER_CLASS=TOTAL_POSTERIOR_SAMPLES//6
assert TOTAL_POSTERIOR_SAMPLES%6==0

BEST_DPS={"zeta":0.05,"apply_below":600,"max_ratio":0.50}

N_STATIONS=64
DX=100.0
DZ=100.0
STRIKE_WIDTH=1000.0
RECEIVER_HEIGHT=1.0
NOISE_FRACTION=0.02
DENSITY_LOWER=-1.0
DENSITY_UPPER=1.0

GEOLOGY_TYPES=["blobs","horizontal_layers","dipping_layers","faulted_layers","dykes","intrusions"]
LABEL_TO_ID={name:i for i,name in enumerate(GEOLOGY_TYPES)}
ID_TO_LABEL={i:name for name,i in LABEL_TO_ID.items()}
N_CLASSES=len(GEOLOGY_TYPES)

print("Device:",DEVICE)
print("Total posterior samples per method:",TOTAL_POSTERIOR_SAMPLES)
print("Equal mixture samples per class:",SAMPLES_PER_CLASS)
print("Geology classes:",LABEL_TO_ID)

In [ ]:
# Cell 2 — Load existing simplified datasets and unconditional model
required=[
    f"{DATA_DIR}/train.npy",
    f"{DATA_DIR}/validation.npy",
    f"{DATA_DIR}/test_id.npy",
    f"{DATA_DIR}/test_ood.npy",
    f"{DATA_DIR}/metadata.csv",
    UNCONDITIONAL_MODEL_PATH
]
missing=[p for p in required if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        "Run the simplified-rich end-to-end pipeline first. Missing:\n" + "\n".join(missing)
    )

X_TRAIN=np.load(f"{DATA_DIR}/train.npy",mmap_mode="r")
X_VAL=np.load(f"{DATA_DIR}/validation.npy").astype(np.float32)
X_TEST_ID=np.load(f"{DATA_DIR}/test_id.npy").astype(np.float32)
X_TEST_OOD=np.load(f"{DATA_DIR}/test_ood.npy").astype(np.float32)
METADATA=pd.read_csv(f"{DATA_DIR}/metadata.csv")

TRAIN_META=METADATA[METADATA.split=="train"].reset_index(drop=True)
VAL_META=METADATA[METADATA.split=="validation"].reset_index(drop=True)
TEST_ID_META=METADATA[METADATA.split=="test_id"].reset_index(drop=True)
TEST_OOD_META=METADATA[METADATA.split=="test_ood"].reset_index(drop=True)

Y_TRAIN=np.array([LABEL_TO_ID[x] for x in TRAIN_META.geology_type],dtype=np.int64)
Y_VAL=np.array([LABEL_TO_ID[x] for x in VAL_META.geology_type],dtype=np.int64)
Y_TEST_ID=np.array([LABEL_TO_ID[x] for x in TEST_ID_META.geology_type],dtype=np.int64)
Y_TEST_OOD=np.array([LABEL_TO_ID[x] for x in TEST_OOD_META.geology_type],dtype=np.int64)

assert len(X_TRAIN)==len(Y_TRAIN)
assert len(X_VAL)==len(Y_VAL)
assert len(X_TEST_ID)==len(Y_TEST_ID)
assert len(X_TEST_OOD)==len(Y_TEST_OOD)

print("Train:",X_TRAIN.shape)
print("Validation:",X_VAL.shape)
print("ID:",X_TEST_ID.shape)
print("OOD:",X_TEST_OOD.shape)

In [ ]:
# Cell 3 — Unconditional and conditional U-Net architectures
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self,dim):
        super().__init__(); self.dim=dim
        self.mlp=nn.Sequential(nn.Linear(dim,dim*4),nn.SiLU(),nn.Linear(dim*4,dim))
    def forward(self,t):
        half=self.dim//2
        freqs=torch.exp(-math.log(10000)*torch.arange(half,device=t.device)/(half-1))
        args=t[:,None].float()*freqs[None,:]
        return self.mlp(torch.cat([args.sin(),args.cos()],dim=-1))

class ResBlock(nn.Module):
    def __init__(self,in_ch,out_ch,conditioning_dim):
        super().__init__()
        self.norm1=nn.GroupNorm(8,in_ch); self.conv1=nn.Conv2d(in_ch,out_ch,3,padding=1)
        self.norm2=nn.GroupNorm(8,out_ch); self.conv2=nn.Conv2d(out_ch,out_ch,3,padding=1)
        self.act=nn.SiLU(); self.cond_proj=nn.Linear(conditioning_dim,out_ch)
        self.skip=nn.Conv2d(in_ch,out_ch,1) if in_ch!=out_ch else nn.Identity()
    def forward(self,x,conditioning):
        h=self.conv1(self.act(self.norm1(x)))
        h=h+self.cond_proj(self.act(conditioning))[:,:,None,None]
        return self.conv2(self.act(self.norm2(h)))+self.skip(x)

class DownBlock(nn.Module):
    def __init__(self,in_ch,out_ch,conditioning_dim):
        super().__init__(); self.res=ResBlock(in_ch,out_ch,conditioning_dim); self.down=nn.Conv2d(out_ch,out_ch,4,2,1)
    def forward(self,x,conditioning):
        x=self.res(x,conditioning); return self.down(x),x

class UpBlock(nn.Module):
    def __init__(self,in_ch,skip_ch,out_ch,conditioning_dim):
        super().__init__(); self.up=nn.ConvTranspose2d(in_ch,in_ch,4,2,1)
        self.res=ResBlock(in_ch+skip_ch,out_ch,conditioning_dim)
    def forward(self,x,skip,conditioning):
        return self.res(torch.cat([self.up(x),skip],dim=1),conditioning)

class UNet(nn.Module):
    def __init__(self,base_ch=32,time_dim=128):
        super().__init__()
        self.time_emb=SinusoidalTimeEmbedding(time_dim)
        self.init_conv=nn.Conv2d(1,base_ch,3,padding=1)
        self.down1=DownBlock(base_ch,base_ch*2,time_dim)
        self.down2=DownBlock(base_ch*2,base_ch*4,time_dim)
        self.bottleneck=ResBlock(base_ch*4,base_ch*4,time_dim)
        self.up2=UpBlock(base_ch*4,base_ch*4,base_ch*2,time_dim)
        self.up1=UpBlock(base_ch*2,base_ch*2,base_ch,time_dim)
        self.out_norm=nn.GroupNorm(8,base_ch); self.out_conv=nn.Conv2d(base_ch,1,3,padding=1); self.act=nn.SiLU()
    def forward(self,x,t):
        conditioning=self.time_emb(t)
        x=self.init_conv(x); x,s1=self.down1(x,conditioning); x,s2=self.down2(x,conditioning)
        x=self.bottleneck(x,conditioning); x=self.up2(x,s2,conditioning); x=self.up1(x,s1,conditioning)
        return self.out_conv(self.act(self.out_norm(x)))

class ConditionalUNet(nn.Module):
    def __init__(self,n_classes=6,base_ch=32,time_dim=128,label_dim=128):
        super().__init__()
        self.time_emb=SinusoidalTimeEmbedding(time_dim)
        self.label_emb=nn.Embedding(n_classes,label_dim)
        self.conditioning=nn.Sequential(
            nn.Linear(time_dim+label_dim,time_dim),
            nn.SiLU(),
            nn.Linear(time_dim,time_dim)
        )
        self.init_conv=nn.Conv2d(1,base_ch,3,padding=1)
        self.down1=DownBlock(base_ch,base_ch*2,time_dim)
        self.down2=DownBlock(base_ch*2,base_ch*4,time_dim)
        self.bottleneck=ResBlock(base_ch*4,base_ch*4,time_dim)
        self.up2=UpBlock(base_ch*4,base_ch*4,base_ch*2,time_dim)
        self.up1=UpBlock(base_ch*2,base_ch*2,base_ch,time_dim)
        self.out_norm=nn.GroupNorm(8,base_ch); self.out_conv=nn.Conv2d(base_ch,1,3,padding=1); self.act=nn.SiLU()
    def forward(self,x,t,labels):
        conditioning=self.conditioning(torch.cat([self.time_emb(t),self.label_emb(labels)],dim=1))
        x=self.init_conv(x); x,s1=self.down1(x,conditioning); x,s2=self.down2(x,conditioning)
        x=self.bottleneck(x,conditioning); x=self.up2(x,s2,conditioning); x=self.up1(x,s1,conditioning)
        return self.out_conv(self.act(self.out_norm(x)))

betas=torch.linspace(1e-4,2e-2,T_STEPS,device=DEVICE)
alphas=1-betas
alpha_bar=torch.cumprod(alphas,0)
sqrt_alpha_bar=alpha_bar.sqrt()
sqrt_one_minus_alpha_bar=(1-alpha_bar).sqrt()
alpha_bar_prev=torch.cat([torch.ones(1,device=DEVICE),alpha_bar[:-1]])
posterior_variance=betas*(1-alpha_bar_prev)/(1-alpha_bar)

unconditional_model=UNet(BASE_CHANNELS,TIME_DIM).to(DEVICE)
unconditional_model.load_state_dict(torch.load(UNCONDITIONAL_MODEL_PATH,map_location=DEVICE,weights_only=True),strict=True)
unconditional_model.eval()
print("Unconditional model loaded.")

In [ ]:
# Cell 4 — Train or resume the conditional DDPM
train_tensor=torch.from_numpy(np.asarray(X_TRAIN)[:,None,:,:])
val_tensor=torch.from_numpy(X_VAL[:,None,:,:])
train_labels=torch.from_numpy(Y_TRAIN)
val_labels=torch.from_numpy(Y_VAL)

train_loader=DataLoader(
    TensorDataset(train_tensor,train_labels),
    batch_size=BATCH_SIZE,shuffle=True,num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),persistent_workers=NUM_WORKERS>0,drop_last=True
)
val_loader=DataLoader(
    TensorDataset(val_tensor,val_labels),
    batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),persistent_workers=NUM_WORKERS>0
)

conditional_model=ConditionalUNet(N_CLASSES,BASE_CHANNELS,TIME_DIM,LABEL_DIM).to(DEVICE)
optimizer=torch.optim.AdamW(conditional_model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer,mode="min",factor=0.5,patience=15,min_lr=1e-6)
scaler=torch.amp.GradScaler("cuda",enabled=torch.cuda.is_available())

RESUME_PATH=f"{TRAIN_DIR}/conditional_resume.pt"
BEST_MODEL_PATH=f"{TRAIN_DIR}/conditional_best.pt"
FINAL_MODEL_PATH=f"{TRAIN_DIR}/conditional_final.pt"
HISTORY_PATH=f"{TRAIN_DIR}/conditional_history.csv"

if FORCE_RETRAIN_CONDITIONAL:
    for p in glob.glob(f"{TRAIN_DIR}/*.pt")+glob.glob(f"{TRAIN_DIR}/*.csv"): os.remove(p)

start_epoch=0; best_val=float("inf"); history=[]
if os.path.exists(RESUME_PATH):
    checkpoint=torch.load(RESUME_PATH,map_location=DEVICE,weights_only=False)
    conditional_model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    start_epoch=int(checkpoint["epoch"])+1
    best_val=float(checkpoint["best_validation_loss"])
    history=checkpoint.get("history",[])
    print("Resuming conditional training from epoch",start_epoch)
elif os.path.exists(BEST_MODEL_PATH):
    conditional_model.load_state_dict(torch.load(BEST_MODEL_PATH,map_location=DEVICE,weights_only=True))
    start_epoch=EPOCHS
    print("Existing conditional best model found; training skipped.")
else:
    print("Training conditional DDPM from scratch.")

@torch.no_grad()
def sample_conditional_prior(model,n_samples,label_id,seed):
    generator=torch.Generator(device=DEVICE).manual_seed(seed)
    x=torch.randn((n_samples,1,GRID_SIZE,GRID_SIZE),generator=generator,device=DEVICE)
    labels=torch.full((n_samples,),label_id,dtype=torch.long,device=DEVICE)
    model.eval()
    for t in range(T_STEPS-1,-1,-1):
        tb=torch.full((n_samples,),t,dtype=torch.long,device=DEVICE)
        eps=model(x,tb,labels)
        mean=(x-betas[t]*eps/sqrt_one_minus_alpha_bar[t].clamp_min(1e-8))/alphas[t].sqrt()
        x=mean+(posterior_variance[t].clamp_min(0).sqrt()*torch.randn(x.shape,generator=generator,device=DEVICE) if t>0 else 0)
    return x.clamp(-1,1).cpu().numpy()[:,0]

def save_conditional_preview(epoch):
    fig,axes=plt.subplots(N_CLASSES,4,figsize=(10,14))
    for label_id in range(N_CLASSES):
        samples=sample_conditional_prior(conditional_model,4,label_id,SEED+epoch*100+label_id)
        for j in range(4):
            axes[label_id,j].imshow(samples[j],cmap="RdBu_r",vmin=-1,vmax=1,origin="upper",aspect="auto")
            axes[label_id,j].set_xticks([]); axes[label_id,j].set_yticks([])
            if j==0: axes[label_id,j].set_ylabel(ID_TO_LABEL[label_id].replace("_"," "))
    plt.suptitle(f"Conditional DDPM samples — epoch {epoch}")
    plt.tight_layout()
    plt.savefig(f"{TRAIN_DIR}/conditional_samples_epoch_{epoch:03d}.png",dpi=160,bbox_inches="tight")
    plt.show()

for epoch in range(start_epoch,EPOCHS):
    start=time.time(); conditional_model.train(); train_sum=0.0
    for x0,labels in train_loader:
        x0=x0.to(DEVICE,non_blocking=True); labels=labels.to(DEVICE,non_blocking=True)
        t=torch.randint(0,T_STEPS,(x0.shape[0],),device=DEVICE)
        noise=torch.randn_like(x0)
        xt=sqrt_alpha_bar[t][:,None,None,None]*x0+sqrt_one_minus_alpha_bar[t][:,None,None,None]*noise
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=torch.cuda.is_available()):
            loss=F.mse_loss(conditional_model(xt,t,labels),noise)
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(conditional_model.parameters(),1.0)
        scaler.step(optimizer); scaler.update(); train_sum+=loss.item()

    conditional_model.eval(); val_sum=0.0
    with torch.no_grad():
        for batch_index,(x0,labels) in enumerate(val_loader):
            x0=x0.to(DEVICE,non_blocking=True); labels=labels.to(DEVICE,non_blocking=True)
            generator=torch.Generator(device=DEVICE).manual_seed(SEED+epoch*100000+batch_index)
            t=torch.randint(0,T_STEPS,(x0.shape[0],),generator=generator,device=DEVICE)
            noise=torch.randn(x0.shape,generator=generator,device=DEVICE,dtype=x0.dtype)
            xt=sqrt_alpha_bar[t][:,None,None,None]*x0+sqrt_one_minus_alpha_bar[t][:,None,None,None]*noise
            val_sum+=F.mse_loss(conditional_model(xt,t,labels),noise).item()

    train_loss=train_sum/len(train_loader); val_loss=val_sum/len(val_loader)
    scheduler.step(val_loss); improved=val_loss<best_val
    if improved:
        best_val=val_loss
        torch.save(conditional_model.state_dict(),BEST_MODEL_PATH)

    history.append({"epoch":epoch+1,"train_loss":train_loss,"validation_loss":val_loss,"learning_rate":optimizer.param_groups[0]["lr"],"epoch_seconds":time.time()-start})
    pd.DataFrame(history).to_csv(HISTORY_PATH,index=False)
    checkpoint={
        "epoch":epoch,
        "model_state_dict":conditional_model.state_dict(),
        "optimizer_state_dict":optimizer.state_dict(),
        "scheduler_state_dict":scheduler.state_dict(),
        "scaler_state_dict":scaler.state_dict(),
        "best_validation_loss":best_val,
        "history":history
    }
    torch.save(checkpoint,RESUME_PATH)
    if (epoch+1)%CHECKPOINT_EVERY==0:
        torch.save(checkpoint,f"{TRAIN_DIR}/conditional_checkpoint_{epoch+1:03d}.pt")

    print(f"Epoch {epoch+1:3d}/{EPOCHS} | train={train_loss:.6f} | val={val_loss:.6f} | best={best_val:.6f} | time={time.time()-start:.1f}s{' | BEST' if improved else ''}")
    if (epoch+1)%SAMPLE_EVERY==0 or epoch==start_epoch:
        save_conditional_preview(epoch+1)

torch.save(conditional_model.state_dict(),FINAL_MODEL_PATH)
conditional_model.load_state_dict(torch.load(BEST_MODEL_PATH,map_location=DEVICE,weights_only=True))
conditional_model.eval()
print("Conditional best model loaded:",BEST_MODEL_PATH)

In [ ]:
# Cell 5 — Shared SimPEG gravity operator
hx=np.full(GRID_SIZE,DX); hy=np.array([STRIKE_WIDTH]); hz=np.full(GRID_SIZE,DZ)
mesh=TensorMesh([hx,hy,hz],x0=[-DX/2,-STRIKE_WIDTH/2,-GRID_SIZE*DZ])
receiver_x=np.linspace(0,(GRID_SIZE-1)*DX,N_STATIONS)
receiver_locations=np.c_[receiver_x,np.zeros(N_STATIONS),np.full(N_STATIONS,RECEIVER_HEIGHT)]
receiver=gravity.receivers.Point(receiver_locations,components="gz")
source=gravity.sources.SourceField(receiver_list=[receiver])
survey=gravity.survey.Survey(source)
active=np.ones(mesh.nC,dtype=bool)
model_map=maps.IdentityMap(nP=mesh.nC)
simulation=gravity.simulation.Simulation3DIntegral(
    mesh=mesh,survey=survey,rhoMap=model_map,active_cells=active,
    store_sensitivities="ram",sensitivity_dtype=np.float32,engine="choclo"
)
_ = simulation.dpred(np.zeros(mesh.nC))
G_mesh=np.asarray(simulation.G,dtype=np.float32)
perm=np.arange(GRID_SIZE*GRID_SIZE).reshape(GRID_SIZE,GRID_SIZE)[::-1,:].reshape(-1)
G_img=G_mesh[:,perm]
G_torch=torch.tensor(G_img,dtype=torch.float32,device=DEVICE)

def image_to_mesh(m): return np.asarray(m,dtype=np.float64)[::-1,:].reshape(-1)
def mesh_to_image(m): return np.asarray(m).reshape(GRID_SIZE,GRID_SIZE)[::-1,:].astype(np.float32)

check=np.random.default_rng(1).normal(size=(GRID_SIZE,GRID_SIZE))
assert np.allclose(simulation.dpred(image_to_mesh(check)),G_img@check.ravel(),rtol=1e-5,atol=1e-7)
print("SimPEG operator ready:",G_img.shape)

In [ ]:
# Cell 6 — Observation, SimPEG L2 and DPS samplers
def make_observation(rho_image,seed):
    clean=simulation.dpred(image_to_mesh(rho_image)).astype(np.float64)
    sigma=float(NOISE_FRACTION*np.max(np.abs(clean)))
    rng=np.random.default_rng(seed); noise=rng.normal(0,sigma,size=clean.size)
    return clean+noise,clean,np.full(clean.size,sigma)

def run_simpeg_l2(dobs,std):
    d_obj=data.Data(survey,dobs=dobs,standard_deviation=std)
    dmis=data_misfit.L2DataMisfit(data=d_obj,simulation=simulation)
    reg=regularization.WeightedLeastSquares(mesh,active_cells=active,mapping=model_map)
    opt=optimization.ProjectedGNCG(maxIter=20,lower=DENSITY_LOWER,upper=DENSITY_UPPER,maxIterLS=20,maxIterCG=30,tolCG=1e-4)
    inv_prob=inverse_problem.BaseInvProblem(dmis,reg,opt)
    directive_list=[
        directives.UpdateSensitivityWeights(every_iteration=False),
        directives.BetaEstimate_ByEig(beta0_ratio=10.0),
        directives.BetaSchedule(coolingFactor=5.0,coolingRate=1),
        directives.TargetMisfit(chifact=1.0),
        directives.UpdatePreconditioner()
    ]
    buffer=io.StringIO()
    with contextlib.redirect_stdout(buffer),contextlib.redirect_stderr(buffer):
        recovered=inversion.BaseInversion(inv_prob,directiveList=directive_list).run(np.zeros(mesh.nC))
    return mesh_to_image(recovered),simulation.dpred(recovered)

def straight_through_clip(x,lo=-1.,hi=1.):
    clipped=x.clamp(lo,hi)
    return x+(clipped-x).detach()

def dps_unconditional(model,g_obs,n_samples,seed,zeta=.05,apply_below=600,max_ratio=.5):
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    obs=torch.tensor(g_obs,dtype=torch.float32,device=DEVICE)[None].expand(n_samples,-1)
    x=torch.randn(n_samples,1,GRID_SIZE,GRID_SIZE,device=DEVICE)

    for t in range(T_STEPS-1,-1,-1):
        apply=t<apply_below
        xc=x.detach().requires_grad_(apply)
        tb=torch.full((n_samples,),t,dtype=torch.long,device=DEVICE)
        with torch.set_grad_enabled(apply):
            eps=model(xc,tb)
            x0=straight_through_clip((xc-sqrt_one_minus_alpha_bar[t]*eps)/sqrt_alpha_bar[t].clamp_min(1e-6))
            mean=(xc-betas[t]/sqrt_one_minus_alpha_bar[t].clamp_min(1e-6)*eps)/alphas[t].sqrt()
            xp=mean+(posterior_variance[t].clamp_min(0).sqrt()*torch.randn_like(mean) if t>0 else 0)
        prior=xp.detach()-xc.detach()
        prior_rms=prior.square().mean((1,2,3),keepdim=True).sqrt()
        if apply:
            pred=(G_torch@x0.reshape(n_samples,-1).T).T
            loss=(pred-obs).square().mean(1).sum()
            grad=torch.autograd.grad(loss,xc)[0]
            grad_rms=grad.square().mean((1,2,3),keepdim=True).sqrt().clamp_min(1e-8)
            raw=torch.full_like(prior_rms,float(zeta*sqrt_one_minus_alpha_bar[t]))
            used=torch.minimum(raw,max_ratio*prior_rms.clamp_min(1e-4))
            xp=xp.detach()-used*grad.detach()/grad_rms
        x=xp.detach()
    return x.clamp(-1,1).detach()

def dps_conditional(model,g_obs,labels,seed,zeta=.05,apply_below=600,max_ratio=.5):
    labels=torch.as_tensor(labels,dtype=torch.long,device=DEVICE)
    n_samples=len(labels)
    torch.manual_seed(seed); np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    obs=torch.tensor(g_obs,dtype=torch.float32,device=DEVICE)[None].expand(n_samples,-1)
    x=torch.randn(n_samples,1,GRID_SIZE,GRID_SIZE,device=DEVICE)

    for t in range(T_STEPS-1,-1,-1):
        apply=t<apply_below
        xc=x.detach().requires_grad_(apply)
        tb=torch.full((n_samples,),t,dtype=torch.long,device=DEVICE)
        with torch.set_grad_enabled(apply):
            eps=model(xc,tb,labels)
            x0=straight_through_clip((xc-sqrt_one_minus_alpha_bar[t]*eps)/sqrt_alpha_bar[t].clamp_min(1e-6))
            mean=(xc-betas[t]/sqrt_one_minus_alpha_bar[t].clamp_min(1e-6)*eps)/alphas[t].sqrt()
            xp=mean+(posterior_variance[t].clamp_min(0).sqrt()*torch.randn_like(mean) if t>0 else 0)
        prior=xp.detach()-xc.detach()
        prior_rms=prior.square().mean((1,2,3),keepdim=True).sqrt()
        if apply:
            pred=(G_torch@x0.reshape(n_samples,-1).T).T
            loss=(pred-obs).square().mean(1).sum()
            grad=torch.autograd.grad(loss,xc)[0]
            grad_rms=grad.square().mean((1,2,3),keepdim=True).sqrt().clamp_min(1e-8)
            raw=torch.full_like(prior_rms,float(zeta*sqrt_one_minus_alpha_bar[t]))
            used=torch.minimum(raw,max_ratio*prior_rms.clamp_min(1e-4))
            xp=xp.detach()-used*grad.detach()/grad_rms
        x=xp.detach()
    return x.clamp(-1,1).detach()

print("All inversion methods ready.")

In [ ]:
# Cell 7 — Resumable benchmark: SimPEG, unconditional, oracle conditional and equal mixture
if FORCE_REBENCHMARK:
    for p in glob.glob(f"{CASE_DIR}/*.npz"): os.remove(p)

def case_path(split,index): return f"{CASE_DIR}/{split}_{index:03d}.npz"

def summarize_samples(samples,truth,obs):
    mean=samples.mean(0)
    std=samples.std(0,ddof=1)
    prediction=G_img@mean.ravel()
    return {
        "mean":mean,
        "std":std,
        "prediction":prediction,
        "model_rmse":float(np.sqrt(np.mean((mean-truth)**2))),
        "correlation":float(np.corrcoef(mean.ravel(),truth.ravel())[0,1]),
        "data_rmse":float(np.sqrt(np.mean((prediction-obs)**2))),
        "uncertainty_error_spearman":float(spearmanr(std.ravel(),np.abs(mean-truth).ravel(),nan_policy="omit").statistic),
        "q16":np.quantile(samples,.16,axis=0),
        "q84":np.quantile(samples,.84,axis=0),
        "q025":np.quantile(samples,.025,axis=0),
        "q975":np.quantile(samples,.975,axis=0)
    }

def run_case(split,index,truth,true_label,seed_offset):
    obs,clean,std=make_observation(truth,seed_offset+index)

    start=time.time()
    l2,l2_pred=run_simpeg_l2(obs,std)
    l2_runtime=time.time()-start

    start=time.time()
    uncond=dps_unconditional(unconditional_model,obs,TOTAL_POSTERIOR_SAMPLES,seed_offset+10000+index,**BEST_DPS)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    uncond_runtime=time.time()-start
    uncond_samples=uncond[:,0].cpu().numpy()
    uncond_summary=summarize_samples(uncond_samples,truth,obs)

    oracle_labels=np.full(TOTAL_POSTERIOR_SAMPLES,true_label,dtype=np.int64)
    start=time.time()
    oracle=dps_conditional(conditional_model,obs,oracle_labels,seed_offset+20000+index,**BEST_DPS)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    oracle_runtime=time.time()-start
    oracle_samples=oracle[:,0].cpu().numpy()
    oracle_summary=summarize_samples(oracle_samples,truth,obs)

    mixture_labels=np.repeat(np.arange(N_CLASSES),SAMPLES_PER_CLASS)
    start=time.time()
    mixture=dps_conditional(conditional_model,obs,mixture_labels,seed_offset+30000+index,**BEST_DPS)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    mixture_runtime=time.time()-start
    mixture_samples=mixture[:,0].cpu().numpy()
    mixture_summary=summarize_samples(mixture_samples,truth,obs)

    def coverage(summary):
        return (
            float(np.mean((truth>=summary["q16"])&(truth<=summary["q84"]))),
            float(np.mean((truth>=summary["q025"])&(truth<=summary["q975"])))
        )

    uncond_cov68,uncond_cov95=coverage(uncond_summary)
    oracle_cov68,oracle_cov95=coverage(oracle_summary)
    mixture_cov68,mixture_cov95=coverage(mixture_summary)

    l2_rmse=float(np.sqrt(np.mean((l2-truth)**2)))
    metrics={
        "split":split,
        "case":index,
        "geology_type":ID_TO_LABEL[int(true_label)],
        "l2_model_rmse":l2_rmse,
        "l2_correlation":float(np.corrcoef(l2.ravel(),truth.ravel())[0,1]),
        "l2_data_rmse":float(np.sqrt(np.mean((l2_pred-obs)**2))),
        "unconditional_model_rmse":uncond_summary["model_rmse"],
        "unconditional_correlation":uncond_summary["correlation"],
        "unconditional_data_rmse":uncond_summary["data_rmse"],
        "unconditional_uncertainty_error_spearman":uncond_summary["uncertainty_error_spearman"],
        "unconditional_coverage_68":uncond_cov68,
        "unconditional_coverage_95":uncond_cov95,
        "oracle_model_rmse":oracle_summary["model_rmse"],
        "oracle_correlation":oracle_summary["correlation"],
        "oracle_data_rmse":oracle_summary["data_rmse"],
        "oracle_uncertainty_error_spearman":oracle_summary["uncertainty_error_spearman"],
        "oracle_coverage_68":oracle_cov68,
        "oracle_coverage_95":oracle_cov95,
        "mixture_model_rmse":mixture_summary["model_rmse"],
        "mixture_correlation":mixture_summary["correlation"],
        "mixture_data_rmse":mixture_summary["data_rmse"],
        "mixture_uncertainty_error_spearman":mixture_summary["uncertainty_error_spearman"],
        "mixture_coverage_68":mixture_cov68,
        "mixture_coverage_95":mixture_cov95,
        "l2_runtime_seconds":l2_runtime,
        "unconditional_runtime_seconds":uncond_runtime,
        "oracle_runtime_seconds":oracle_runtime,
        "mixture_runtime_seconds":mixture_runtime
    }
    metrics["unconditional_wins_l2"]=metrics["unconditional_model_rmse"]<l2_rmse
    metrics["oracle_wins_l2"]=metrics["oracle_model_rmse"]<l2_rmse
    metrics["mixture_wins_l2"]=metrics["mixture_model_rmse"]<l2_rmse
    metrics["oracle_improves_unconditional"]=metrics["oracle_model_rmse"]<metrics["unconditional_model_rmse"]
    metrics["mixture_improves_unconditional"]=metrics["mixture_model_rmse"]<metrics["unconditional_model_rmse"]

    np.savez_compressed(
        case_path(split,index),
        metrics=np.array(metrics,dtype=object),
        truth=truth.astype(np.float32),
        l2=l2.astype(np.float32),
        unconditional_mean=uncond_summary["mean"].astype(np.float32),
        unconditional_std=uncond_summary["std"].astype(np.float32),
        oracle_mean=oracle_summary["mean"].astype(np.float32),
        oracle_std=oracle_summary["std"].astype(np.float32),
        mixture_mean=mixture_summary["mean"].astype(np.float32),
        mixture_std=mixture_summary["std"].astype(np.float32),
        observations=obs.astype(np.float32)
    )

    del uncond,oracle,mixture
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def run_split(split,models,labels,seed_offset):
    pending=[i for i in range(N_TEST_PER_SPLIT) if not os.path.exists(case_path(split,i))]
    print(f"{split.upper()}: {N_TEST_PER_SPLIT-len(pending)} completed, {len(pending)} remaining")
    for i in tqdm(pending,desc=f"{split.upper()} conditional benchmark"):
        run_case(split,i,models[i],labels[i],seed_offset)

run_split("id",X_TEST_ID,Y_TEST_ID,50000)
run_split("ood",X_TEST_OOD,Y_TEST_OOD,70000)

In [ ]:
# Cell 8 — Summaries, statistical tests and figures
def load_split(split):
    files=[case_path(split,i) for i in range(N_TEST_PER_SPLIT)]
    missing=[p for p in files if not os.path.exists(p)]
    if missing: raise RuntimeError(f"{len(missing)} {split} cases missing; rerun Cell 7.")
    loaded=[np.load(p,allow_pickle=True) for p in files]
    results=pd.DataFrame([x["metrics"].item() for x in loaded])
    arrays={key:np.stack([x[key] for x in loaded]) for key in [
        "truth","l2","unconditional_mean","unconditional_std",
        "oracle_mean","oracle_std","mixture_mean","mixture_std","observations"
    ]}
    results.to_csv(f"{RESULTS_DIR}/{split}_benchmark.csv",index=False)
    np.savez_compressed(f"{RESULTS_DIR}/{split}_arrays.npz",**arrays)
    return results,arrays

id_results,id_arrays=load_split("id")
ood_results,ood_arrays=load_split("ood")
all_results=pd.concat([id_results,ood_results],ignore_index=True)
all_results.to_csv(f"{RESULTS_DIR}/all_benchmark.csv",index=False)

overall=all_results.groupby("split").agg(
    n=("case","count"),
    l2_rmse=("l2_model_rmse","mean"),
    unconditional_rmse=("unconditional_model_rmse","mean"),
    oracle_rmse=("oracle_model_rmse","mean"),
    equal_mixture_rmse=("mixture_model_rmse","mean"),
    l2_correlation=("l2_correlation","mean"),
    unconditional_correlation=("unconditional_correlation","mean"),
    oracle_correlation=("oracle_correlation","mean"),
    equal_mixture_correlation=("mixture_correlation","mean"),
    unconditional_win_rate=("unconditional_wins_l2","mean"),
    oracle_win_rate=("oracle_wins_l2","mean"),
    equal_mixture_win_rate=("mixture_wins_l2","mean"),
    oracle_beats_unconditional=("oracle_improves_unconditional","mean"),
    equal_mixture_beats_unconditional=("mixture_improves_unconditional","mean"),
    unconditional_coverage_68=("unconditional_coverage_68","mean"),
    oracle_coverage_68=("oracle_coverage_68","mean"),
    mixture_coverage_68=("mixture_coverage_68","mean"),
    unconditional_coverage_95=("unconditional_coverage_95","mean"),
    oracle_coverage_95=("oracle_coverage_95","mean"),
    mixture_coverage_95=("mixture_coverage_95","mean")
).reset_index()

by_type=all_results.groupby(["split","geology_type"]).agg(
    n=("case","count"),
    l2_rmse=("l2_model_rmse","mean"),
    unconditional_rmse=("unconditional_model_rmse","mean"),
    oracle_rmse=("oracle_model_rmse","mean"),
    equal_mixture_rmse=("mixture_model_rmse","mean"),
    unconditional_win_rate=("unconditional_wins_l2","mean"),
    oracle_win_rate=("oracle_wins_l2","mean"),
    equal_mixture_win_rate=("mixture_wins_l2","mean")
).reset_index()

stats=[]
for split,df in [("id",id_results),("ood",ood_results)]:
    comparisons=[
        ("oracle_vs_unconditional",df.oracle_model_rmse.to_numpy(),df.unconditional_model_rmse.to_numpy()),
        ("mixture_vs_unconditional",df.mixture_model_rmse.to_numpy(),df.unconditional_model_rmse.to_numpy()),
        ("unconditional_vs_l2",df.unconditional_model_rmse.to_numpy(),df.l2_model_rmse.to_numpy()),
        ("oracle_vs_l2",df.oracle_model_rmse.to_numpy(),df.l2_model_rmse.to_numpy()),
        ("mixture_vs_l2",df.mixture_model_rmse.to_numpy(),df.l2_model_rmse.to_numpy())
    ]
    for name,a,b in comparisons:
        diff=a-b
        rng=np.random.default_rng(SEED)
        boot=np.array([diff[rng.integers(0,len(diff),len(diff))].mean() for _ in range(5000)])
        stats.append({
            "split":split,
            "comparison":name,
            "mean_difference":float(diff.mean()),
            "ci_low":float(np.quantile(boot,.025)),
            "ci_high":float(np.quantile(boot,.975)),
            "wilcoxon_p":float(wilcoxon(a,b).pvalue),
            "paired_t_p":float(ttest_rel(a,b).pvalue)
        })
stats=pd.DataFrame(stats)

overall.to_csv(f"{RESULTS_DIR}/overall_summary.csv",index=False)
by_type.to_csv(f"{RESULTS_DIR}/summary_by_geology_type.csv",index=False)
stats.to_csv(f"{RESULTS_DIR}/statistical_tests.csv",index=False)

display(overall)
display(by_type)
display(stats)

fig,axes=plt.subplots(1,2,figsize=(13,4.8))
for ax,(title,df) in zip(axes,[("ID",id_results),("OOD",ood_results)]):
    ax.boxplot(
        [df.l2_model_rmse,df.unconditional_model_rmse,df.oracle_model_rmse,df.mixture_model_rmse],
        tick_labels=["SimPEG L2","Unconditional","Oracle conditional","Equal mixture"]
    )
    ax.set_title(title); ax.set_ylabel("Model RMSE"); ax.grid(axis="y",alpha=.3)
    ax.tick_params(axis="x",rotation=20)
plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/method_rmse_comparison.png",dpi=180,bbox_inches="tight")
plt.show()

for split,arrays,results in [("id",id_arrays,id_results),("ood",ood_arrays,ood_results)]:
    improvement=results.unconditional_model_rmse-results.oracle_model_rmse
    for label,case in [("largest_oracle_gain",int(improvement.idxmax())),("smallest_oracle_gain",int(improvement.idxmin()))]:
        truth=arrays["truth"][case]; l2=arrays["l2"][case]
        uncond=arrays["unconditional_mean"][case]; oracle=arrays["oracle_mean"][case]
        mixture=arrays["mixture_mean"][case]; mix_std=arrays["mixture_std"][case]
        vmax=max(np.abs(truth).max(),np.abs(l2).max(),np.abs(uncond).max(),np.abs(oracle).max(),np.abs(mixture).max())
        fig,ax=plt.subplots(1,6,figsize=(20,3.5))
        items=[
            (truth,"Truth","RdBu_r",-vmax,vmax),
            (l2,"SimPEG L2","RdBu_r",-vmax,vmax),
            (uncond,"Unconditional","RdBu_r",-vmax,vmax),
            (oracle,"Oracle conditional","RdBu_r",-vmax,vmax),
            (mixture,"Equal mixture","RdBu_r",-vmax,vmax),
            (mix_std,"Mixture std","magma",0,None)
        ]
        for a,(image,title,cmap,vmin,vmax_local) in zip(ax,items):
            a.imshow(image,cmap=cmap,vmin=vmin,vmax=vmax_local,origin="upper",aspect="auto")
            a.set_title(title); a.set_xticks([]); a.set_yticks([])
        fig.suptitle(f"{split.upper()} {label} — {results.loc[case,'geology_type']}")
        plt.tight_layout()
        plt.savefig(f"{RESULTS_DIR}/{split}_{label}.png",dpi=180,bbox_inches="tight")
        plt.show()

In [ ]:
# Cell 9 — Export lightweight analysis ZIP
metadata={
    "geology_types":GEOLOGY_TYPES,
    "label_to_id":LABEL_TO_ID,
    "total_posterior_samples":TOTAL_POSTERIOR_SAMPLES,
    "samples_per_class_equal_mixture":SAMPLES_PER_CLASS,
    "best_dps":BEST_DPS,
    "n_test_per_split":N_TEST_PER_SPLIT,
    "noise_fraction":NOISE_FRACTION,
    "weighting":"equal"
}
with open(f"{RESULTS_DIR}/metadata.json","w") as f: json.dump(metadata,f,indent=2)

FINAL_ZIP="/kaggle/working/conditional_equal_mixture_results.zip"
with zipfile.ZipFile(FINAL_ZIP,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=9) as archive:
    for root,_,files in os.walk(RESULTS_DIR):
        for filename in files:
            if filename.endswith(".npz") and os.path.basename(root)=="cases":
                continue
            path=os.path.join(root,filename)
            archive.write(path,arcname=os.path.relpath(path,RESULTS_DIR))
    if os.path.exists(HISTORY_PATH):
        archive.write(HISTORY_PATH,arcname="conditional_history.csv")
    for image in glob.glob(f"{TRAIN_DIR}/conditional_samples_epoch_*.png"):
        archive.write(image,arcname=f"training_previews/{os.path.basename(image)}")

print("Conditional study complete.")
print("Download:",FINAL_ZIP)